In [ ]:
!pip install transformers
!pip install datasets
!pip install scikit-learn
!pip install pandas
!pip install torch

In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake["label"] = 0
true["label"] = 1

df = pd.concat([fake, true])

print(df.head())
print(df["label"].value_counts())

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
def tokenize_function(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=128
    )

In [ ]:
train_encodings = tokenize_function(list(train_texts))
test_encodings = tokenize_function(list(test_texts))

In [ ]:
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'label': list(train_labels)
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'label': list(test_labels)
})

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)

In [ ]:
print(classification_report(test_labels, preds))

In [ ]:
cm = confusion_matrix(test_labels, preds)

print("Confusion Matrix:")
print(cm)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(test_labels, preds)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
model.save_pretrained("fake_news_bert_model")
tokenizer.save_pretrained("fake_news_bert_model")

In [ ]:
import torch
import torch.nn.functional as F

def predict_news(text):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}
    model.to(device)

    outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)

    prediction = torch.argmax(probs)

    fake_prob = probs[0][0].item()
    real_prob = probs[0][1].item()

    if prediction == 1:
        print("Prediction: Real News")
    else:
        print("Prediction: Fake News")

    print(f"Fake probability: {fake_prob:.4f}")
    print(f"Real probability: {real_prob:.4f}")

Example test:

In [ ]:
predict_news("Government announces new economic policy today")

In [ ]:
predict_news("Scientists discovered a new cancer treatment")
predict_news("Aliens landed in New York yesterday")
predict_news("Government announces new tax reforms")

In [ ]:
import pandas as pd

# Create dataframe for analysis
analysis_df = pd.DataFrame({
    "text": test_texts,
    "actual_label": test_labels,
    "predicted_label": preds
})

# Find misclassified samples
errors = analysis_df[analysis_df["actual_label"] != analysis_df["predicted_label"]]

print("Total incorrect predictions:", len(errors))

# Show first 5 errors
errors.head(5)

In [ ]:
if len(errors) == 0:
    print("No misclassified samples found. Model predicted all test samples correctly.")
else:
    display(errors.head(5))

Model Improvement step by tuning learning rate

In [ ]:
from transformers import TrainingArguments

training_args_lr = TrainingArguments(
    output_dir="./results_lr",
    learning_rate=5e-5,   # tuned learning rate
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3
)

In [ ]:
trainer_lr = Trainer(
    model=model,
    args=training_args_lr,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer_lr.train()

In [ ]:
predictions_lr = trainer_lr.predict(test_dataset)

preds_lr = predictions_lr.predictions.argmax(axis=1)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_labels, preds_lr))

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_before = accuracy_score(test_labels, preds)
accuracy_after = accuracy_score(test_labels, preds_lr)

print("Accuracy Before Tuning:", accuracy_before)
print("Accuracy After Tuning:", accuracy_after)

In [ ]:
!pip install gradio

In [ ]:
import torch

def predict_news(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits).item()

    if prediction == 0:
        return "Real News"
    else:
        return "Fake News"

In [ ]:
model.eval()

In [ ]:
import gradio as gr

interface = gr.Interface(
    fn=predict_news,
    inputs="text",
    outputs="text",
    title="Fake News Detection",
    description="Enter a news headline or article to check if it is Fake or Real."
)

interface.launch(share=True)